In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

Define your table names

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]

SCHEMA = "skypoints"

BRONZE_TABLE = f"{catalog}.{SCHEMA}.bronze_member_feed"

SILVER_TABLE = f"{catalog}.{SCHEMA}.silver_member"

SILVER_REJECTS = f"{catalog}.{SCHEMA}.silver_member_rejects"

GOLD_MASTER = f"{catalog}.{SCHEMA}.gold_member_master"

GOLD_IND = f"{catalog}.{SCHEMA}.gold_member_ind"

GOLD_USA = f"{catalog}.{SCHEMA}.gold_member_usa"

GOLD_AUS = f"{catalog}.{SCHEMA}.gold_member_aus"

GOLD_REDEMPTION = f"{catalog}.{SCHEMA}.gold_redemption"

REDEMPTION_REJECTS = f"{catalog}.{SCHEMA}.silver_redemption_rejects"


Bronze layer validation

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

bronze_count = bronze_df.count()

print("Bronze record count:", bronze_count)

In [0]:
# Country-wise count

(
    bronze_df
    .groupBy("SOURCE_COUNTRY")
    .count()
    .orderBy("SOURCE_COUNTRY")
    .show()
)

In [0]:
# Null bronze raw records

(
    bronze_df
    .filter(F.col("RAW_RECORD").isNull())
).show()

In [0]:
# Duplicate source records
(
    bronze_df
    .groupBy("SOURCE_COUNTRY","RECORD_HASH")
    .count()
    .filter(F.col("count") > 1)
).show()

Silver layer validation

In [0]:
silver_df = spark.table(SILVER_TABLE)
silver_reject_df = spark.table(SILVER_REJECTS)

In [0]:
silver_count = silver_df.count()
silver_reject_count = silver_reject_df.count()

print("Silver valid records:", silver_count)
print("Silver rejected records:", silver_reject_count)
print("Total Silver processed:", silver_count + silver_reject_count)

In [0]:
# Validate rejected records

display(
    silver_reject_df
    .select(
        "MEMBER_ID",
        "MEMBER_NAME",
        "SOURCE_COUNTRY",
        "ENROLLMENT_DATE_RAW",
        "DQ_ERRORS"
    )
)

In [0]:
# Check mandatory fields in Silver

display(
    silver_df
    .filter(
        F.col("MEMBER_ID").isNull() |
        (F.trim(F.col("MEMBER_ID")) == "") |

        F.col("MEMBER_NAME").isNull() |
        (F.trim(F.col("MEMBER_NAME")) == "") |

        F.col("ENROLLMENT_DATE").isNull()
    )
)

In [0]:
# Check duplicate member keys

display(
    silver_df
    .groupBy("SOURCE_COUNTRY", "MEMBER_ID")
    .count()
    .filter(F.col("count") > 1)
)

Gold layer validation

In [0]:
gold_master_df = spark.table(GOLD_MASTER)

gold_ind_df = spark.table(GOLD_IND)
gold_usa_df = spark.table(GOLD_USA)
gold_aus_df = spark.table(GOLD_AUS)

print("Gold master:", gold_master_df.count())
print("India:", gold_ind_df.count())
print("USA:", gold_usa_df.count())
print("Australia:", gold_aus_df.count())

In [0]:
# Country segregation

(
    gold_master_df
    .groupBy("SOURCE_COUNTRY").count()
    .orderBy("SOURCE_COUNTRY")
).show()

In [0]:
# Gold master reconciliation

country_total = (
    gold_ind_df.count() + 
    gold_usa_df.count() + 
    gold_aus_df.count()
)

master_total = gold_master_df.count()

print("Country table total: ", country_total)
print("Gold master total: ", master_total)


assert country_total == master_total,\
"Gold master and country tables have different record counts"


In [0]:
# Duplicate Gold member key

gold_duplicates = (
    gold_master_df
    .groupBy("MEMBER_KEY").count()
    .filter(F.col("count") > 1)
)

Age and Stale_Member validation

In [0]:
# Age validation
(
    silver_df
    .filter(F.col("DOB").isNull())
).select(
    "MEMBER_ID",
    "MEMBER_NAME",
    "SOURCE_COUNTRY",
    "DOB",
    "AGE"
).show()

In [0]:
# Stale member validation

(
    silver_df.filter(
        (F.datediff(F.current_date(), F.col("FLIGHT_DATE")) > 90) &
        (
            F.col("STALE_MEMBER") != "Y"
        )
    )
).show()

In [0]:
# missing flight dates

(
    silver_df.
    filter(
        (F.col("FLIGHT_DATE").isNull()) &
        (F.col("STALE_MEMBER") != "UNKNOWN")
    )
).show()

Redemption JSON validation

In [0]:
redemption_gold_df = spark.table(GOLD_REDEMPTION)

redemption_reject_df = spark.table(REDEMPTION_REJECTS)

print("Valid redemption records:", redemption_gold_df.count())
print("Rejected redemption records:", redemption_reject_df.count())

In [0]:
# Duplicate transaction IDs
(
    redemption_gold_df
    .groupBy("TXN_ID").count()
    .filter(F.col("count") > 1)
).show()

In [0]:
# Validate miles redeemed

(
    redemption_gold_df
    .filter(
        F.col("MILES_REDEEMED").isNull() |
        (F.col("MILES_REDEEMED") <= 0)
    )
).show()

In [0]:
# Validate transaction status

(
    redemption_gold_df
    .groupBy("STATUS").count()
).show()

In [0]:
# Reconcile total miles

(
    redemption_gold_df.agg(
        F.count("*").alias("TOTAL_TRANSACTIONS"),
        F.sum("MILES_REDEEMED").alias("TOTAL_MILES_REDEEMED")
    )
).show()

In [0]:
bronze_count = spark.table(BRONZE_TABLE).count()
silver_valid_count = spark.table(SILVER_TABLE).count()
silver_rejected_count = spark.table(SILVER_REJECTS).count()
gold_count = spark.table(GOLD_MASTER).count()

print("Bronze:", bronze_count)
print("Silver Valid:", silver_valid_count)
print("Silver Rejected:", silver_rejected_count)
print("Gold:", gold_count)

print(
    "Bronze reconciliation: ",
    bronze_count == silver_valid_count + silver_rejected_count
)

print(
    "Silver to Gold reconciliation: ",
    silver_valid_count == gold_count
)


In [0]:
test_results = []

def run_test(test_name, condition, actual_result):
    status = "PASS" if condition else "FAIL"

    test_results.append((
        test_name,
        status,
        str(actual_result)
    ))

    print(f"{status} | {test_name} | {actual_result}")

In [0]:
run_test(
    "Bronze record count",
    bronze_count == 9,
    bronze_count
)

run_test(
    "Bronze to Silver reconciliation",
    bronze_count == silver_valid_count + silver_rejected_count,
    f"{silver_valid_count} valid + {silver_rejected_count} rejected"
)

run_test(
    "Silver to Gold reconciliation",
    silver_valid_count == gold_count,
    f"Silver={silver_valid_count}, Gold={gold_count}"
)

run_test(
    "Gold country reconciliation",
    country_total == master_total,
    f"Country total={country_total}, Master={master_total}"
)

run_test(
    "No duplicate Gold member keys",
    gold_duplicates.count() == 0,
    gold_duplicates.count()
)

run_test(
    "Redemption transaction count",
    redemption_gold_df.count() == 2,
    redemption_gold_df.count()
)

run_test(
    "Redemption total miles",
    redemption_gold_df.agg(
        F.sum("MILES_REDEEMED")
    ).first()[0] == 17000,
    redemption_gold_df.agg(
        F.sum("MILES_REDEEMED")
    ).first()[0]
)

results_df = spark.createDataFrame(
    test_results,
    ["TEST_NAME", "STATUS", "ACTUAL_RESULT"]
)

display(results_df)

Check the overall result

In [0]:
failed_tests = results_df.filter(
    F.col("STATUS") == "FAIL"
).count()

print("Total tests:", len(test_results))
print("Failed tests:", failed_tests)

if failed_tests > 0:
    raise Exception("Data quality testing failed. Review failed test cases.")

print("All executed data quality tests passed!")